In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


PATH_MACRO = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\state_z_america.xlsx")

DATASETS = [
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Sectors\Per Region\Sector North America.xlsx"),
     "auto", "Sectors — U.S"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Industries\Developped Countries Industry.xlsx"),
     "auto", "Industries — U.S"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Cross Section Countries\Americas CS Country.xlsx"),
     "auto", "Country Cross Section — America"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\North_America_6_Factors_Daily_clean.xlsx"),
     "auto", "Factors — U.S."),
]


OOS_START, OOS_END = pd.Period('2010-01','M'), pd.Period('2024-12','M')  # ensure 5y lookback


def load_macro_states(path_macro: Path) -> pd.DataFrame:
    Z = pd.read_excel(path_macro)
    if "Month" not in Z.columns:
        for cand in ["month","MONTH","Date","Dates"]:
            if cand in Z.columns:
                Z = Z.rename(columns={cand: "Month"})
                break
    Z["Month"] = pd.to_datetime(Z["Month"], errors="coerce").dt.to_period("M")
    Z = Z.set_index("Month").sort_index()
    macro_cols = [c for c in Z.columns if pd.api.types.is_numeric_dtype(Z[c])]
    return Z[macro_cols].astype(float)

Z_ALL = load_macro_states(PATH_MACRO)

def ann_sr(s: pd.Series) -> float:
    s = pd.Series(s).dropna()
    if s.empty: return np.nan
    sig = s.std()
    return (s.mean()/sig)*np.sqrt(252) if sig != 0 else np.nan

def rank_similar(F_full: pd.DataFrame, F_train: pd.DataFrame, cur_per: pd.Period):
    if cur_per not in F_full.index or F_train.empty:
        return []
    x0 = F_full.loc[[cur_per]].dropna(axis=1, how="any")
    if x0.empty: return []
    Xtr = F_train.dropna(how="any")
    common = Xtr.columns.intersection(x0.columns)
    if common.empty or Xtr.empty: return []
    X = Xtr[common].values.astype(float)
    x = x0.iloc[0][common].values.astype(float)
    d = np.linalg.norm(X - x[None,:], axis=1)
    return Xtr.index[np.argsort(d)].tolist()

def make_quantiles(ranked_months, n_quantiles: int):
    n = len(ranked_months)
    if n == 0:
        return {q: [] for q in range(1, n_quantiles+1)}
    buckets = np.array_split(ranked_months, n_quantiles)
    return {i+1: list(b) for i,b in enumerate(buckets)}

def load_assets(path: Path, kind: str, min_coverage: float = 1.0):
    df = pd.read_excel(path)

    if "Dates" in df.columns:
        df["Dates"] = pd.to_datetime(df["Dates"], errors="coerce")
        df = df.dropna(subset=["Dates"]).set_index("Dates").sort_index()
        assets = [c for c in df.columns if c != "Dates" and pd.api.types.is_numeric_dtype(df[c])]
        sec_ret_daily   = df[assets].pct_change(fill_method=None).replace([np.inf,-np.inf], np.nan).dropna(how="all")
        sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0

    else:
        std_cols = ["Mkt-RF","SMB","HML","RMW","CMA","Mom","RF"]
        date_col = next((c for c in ["Date","Dates","DATE","date","dt"] if c in df.columns), None)
        present  = [c for c in std_cols if c in df.columns]
        if date_col and len(present) >= 5:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
            df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
            df[present] = df[present].astype(float) / 100.0
            assets = [c for c in present if c != "RF"]
            sec_ret_daily   = df[assets].replace([np.inf,-np.inf], np.nan).dropna(how="all")
            sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0
        else:
            raise ValueError(f"Unsupported schema in {path.name}")

    need = max(1, int(np.ceil(min_coverage * len(assets))))
    mask_good = sec_ret_monthly.notna().sum(axis=1) >= need
    sec_ret_monthly = sec_ret_monthly.loc[mask_good]
    sector_months   = sec_ret_monthly.index.to_period("M")
    keep = sec_ret_daily.index.to_period("M").isin(sector_months)
    sec_ret_daily = sec_ret_daily.loc[keep]
    sec_ret_monthly.index = sec_ret_monthly.index.to_period("M")

    return sec_ret_daily, sec_ret_monthly, assets, sector_months



def build_quantile_series(path: Path, kind: str, n_quantiles: int):
    sec_ret_daily, sec_ret_monthly, assets, sector_months = load_assets(path, kind)
    Z = Z_ALL.reindex(sector_months).ffill()
    F_full = Z.copy()

    oos_stop  = min(OOS_END, sector_months.max())
    months_bt = sector_months[(sector_months >= OOS_START) & (sector_months <= oos_stop)]

    returns_by_q = {q: [] for q in range(1, n_quantiles+1)}

    for i in range(len(months_bt)-1):
        cur_per, nxt_per = months_bt[i], months_bt[i+1]

        start_train = cur_per - 35
        F_train = F_full.loc[(F_full.index >= start_train) & (F_full.index <= cur_per)].dropna(how="any")

        ranked = rank_similar(F_full, F_train, cur_per)
        if not ranked: 
            continue
        buckets = make_quantiles(ranked, n_quantiles)

        dts = sec_ret_daily.index[sec_ret_daily.index.to_period("M") == nxt_per]
        if len(dts) == 0: 
            continue
        Rm = sec_ret_daily.loc[dts, assets]

        for q in range(1, n_quantiles+1):
            idx_sim = pd.PeriodIndex(buckets[q], freq="M") + 1
            idx_sim = sec_ret_monthly.index.intersection(idx_sim)
            if len(idx_sim) == 0:
                q_daily = pd.Series(0.0, index=dts)
            else:
                signs = {a: np.sign(sec_ret_monthly.loc[idx_sim, a].mean()) for a in assets}
                w = pd.Series(signs, index=assets, dtype=float)
                q_daily = (Rm * w).mean(axis=1)
            returns_by_q[q].extend(q_daily.items())

    q_series = {q: pd.Series(dict(returns_by_q[q])).sort_index() for q in range(1, n_quantiles+1)}
    for q in q_series:
        q_series[q].index = pd.to_datetime(q_series[q].index)

    common_idx = None
    for q in q_series:
        common_idx = q_series[q].index if common_idx is None else common_idx.intersection(q_series[q].index)
    for q in q_series:
        q_series[q] = q_series[q].reindex(common_idx).fillna(0.0)

    long_only = sec_ret_daily[assets].mean(axis=1).reindex(common_idx).fillna(0.0)
    return q_series, long_only


SAVE_DIR = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

quantile_schemes = [4, 3, 2]

for n_q in quantile_schemes:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False)
    axes = axes.ravel()

    for ax, (path, kind, title) in zip(axes, DATASETS):
        q_series, long_only = build_quantile_series(path, kind, n_q)

        for q in range(1, n_q+1):
            sr = ann_sr(q_series[q])
            ax.plot(q_series[q].index, (1+q_series[q]).cumprod(),
                    label=f"Q{q} (SR {sr:.2f})")

        sr_lo = ann_sr(long_only)
        ax.plot(long_only.index, (1+long_only).cumprod(), color="black", ls="--",
                label=f"Long-only (SR {sr_lo:.2f})")
        
        ax.set_title(f"{title} — {n_q} quantiles (3y rolling train)")

        ax.grid(ls=":", alpha=.6)
        ax.legend(fontsize=7, frameon=False)
    plt.suptitle(f"Quantile Portfolios vs Long-only — {n_q} Quantiles\nTraining = Rolling 3 years", y=0.99)

    plt.tight_layout(rect=[0,0,1,0.96])

    save_path = SAVE_DIR / f"quantile_{n_q}_rolling_3.png"
    plt.savefig(save_path, dpi=300)
    plt.close(fig)

    print(f"Saved figure: {save_path}")


Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_4_rolling_3y.png
Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_3_rolling_3y.png
Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_2_rolling_3y.png
